In [1]:
import pandas as pd
import numpy as np
import cv2 as cv
from pathlib import Path 
import seaborn as sns
import matplotlib.pyplot as plt
from StatTools.generators.ndfnoise_generator import ndfnoise
from tqdm import tqdm
import plotly.express as px
import os
import warnings 
import gc

def draw_traj(trajectories,
              frame_shape: tuple,
              thickness: int):
    
    ants_num = trajectories.shape[1]
    bgs = [np.zeros(shape=frame_shape, dtype=np.uint8) for _ in range(ants_num)]
    
    for ant_idx in range(ants_num):
        
        traj_raw = trajectories[:, ant_idx, :]
        
        valid_mask = ~np.any(np.isnan(traj_raw), axis=1)
        traj_clean = traj_raw[valid_mask]
        
        if len(traj_clean) < 2:
            continue
         
        traj_int = np.ascontiguousarray(traj_clean.astype(np.int32))
        
        cv.polylines(bgs[ant_idx], [traj_int], isClosed=False, color=255, thickness=thickness)
    
    return bgs


def old_calc_iou(tracks, thickness,  frame_shape):
    bgs = draw_traj(tracks, frame_shape=frame_shape, thickness = thickness)
    bgs_arr_bool = np.stack(bgs).astype(np.bool)
    union = bgs_arr_bool.sum(axis=0)
    union_mask = union.astype(np.bool)
    atleast_two_track_intersection = union - union_mask
    iou = atleast_two_track_intersection.sum() / union.sum()
    return iou, atleast_two_track_intersection, union


def gen_traj(frame_num: int,
             ants_num: int,
             frame_shape: tuple,
             margin:int = 10,
             hurst_move: float = 0.5,
             hurst_species: float = 0.5, 
             start_point = None):
    

    dx = ndfnoise(shape=(frame_num, ants_num), hurst=[hurst_move, hurst_species], normalize=True, dtype=np.float32)
    dy = ndfnoise(shape=(frame_num, ants_num), hurst=[hurst_move, hurst_species], normalize=True, dtype=np.float32)

    x = dx.round().cumsum(axis=0).astype(np.int32)
    y = dy.round().cumsum(axis=0).astype(np.int32)

    if start_point is None:
        start_x = np.random.randint(margin, frame_shape[1] - margin, (ants_num,))
        start_y = np.random.randint(margin, frame_shape[0] - margin, (ants_num,))
        start_point = np.stack([start_x, start_y], axis=1)

    trajectories = np.stack([x, y], axis=2)
    trajectories = trajectories + start_point

    return trajectories


def calc_iou(tracks, thickness, frame_shape):

    union = np.zeros(frame_shape, dtype=np.uint16)
    canvas = np.zeros(frame_shape, dtype=np.uint8)

    ants_num = tracks.shape[1]

    for ant_idx in range(ants_num):

        traj = tracks[:, ant_idx]

        valid = ~np.any(np.isnan(traj), axis=1)
        traj = traj[valid]

        if len(traj) < 2:
            continue

        canvas.fill(0)

        cv.polylines(
            canvas,
            [np.ascontiguousarray(traj, dtype=np.int32)],
            isClosed=False,
            color=1,
            thickness=thickness,
        )

        union += canvas

    
    union_mask = union > 0
    intersection = union - union_mask
    iou = intersection.sum() / union.sum()

    return iou, intersection, union

In [2]:
species_list = ['frufa', 'pyeensis']
result = []
frame_shape = (800, 1300)
thikness = 10
for species in species_list:
    dir_path = Path(fr'/home/akhiyarov/asp/NMOT/data/{species}')
    files = list(dir_path.glob("*.csv"))
    pbar = tqdm(files)
    for file_path in pbar:
        pbar.set_description(f"Processing {file_path.name}")
        df_test = pd.read_csv(file_path)[['frame', 'track_id', 'x', 'y']]
        df_test = df_test.astype({'x':np.int32, 'y':np.int32})
        # wide = (df_test.set_index(['frame', 'track_id'])
        #         [['x','y']]
        #         .unstack('track_id'))
        # trajectories = wide.to_numpy().reshape(len(wide), -1, 2)
        
        x_wide = df_test.pivot_table(columns='track_id', index='frame', values='x', fill_value=np.nan)
        y_wide = df_test.pivot_table(columns='track_id', index='frame', values='y', fill_value=np.nan)
        trajectories = np.stack([x_wide.values, y_wide.values], axis=-1)
        del df_test
        del x_wide
        del y_wide
        gc.collect()
        frame_thresholds = np.exp(np.arange(np.log(10), np.log(trajectories.shape[0]), 0.5)).astype(np.int32)
        
        try:
            for frame in frame_thresholds:
                # track_slice = trajectories[:frame]
                iou, _, _  = calc_iou(trajectories[:frame], thikness, frame_shape)

                diff = np.diff(trajectories[:frame], axis=0)

                total_l1_path = np.nansum(np.abs(diff))
                total_l2_path = np.nansum(np.linalg.norm(diff, axis=2))
                del diff
                result.append({'species': species,
                                'name': file_path.name,
                                'frame_threshold':frame,
                                'total_l1_path':total_l1_path,
                                'total_l2_path':total_l2_path,
                                'iou': iou,
                            })
        except Exception as e:
            warnings.warn(f"Ошибка в {file_path.name}: {e}")
            continue

Processing S2190009.csv: 100%|██████████| 45/45 [05:30<00:00,  7.34s/it]  


In [4]:
df_res = pd.DataFrame(result)

In [ ]:
df_res.to_pickle('track_length_iou_estimation.pkl')

In [6]:
px.scatter(df_res, x='total_l1_path', y='iou', color='species')